# Lab 15-08 · PID Control

Four parts, each built on `pid.py`. **Predict first**, then run.

In [1]:
import math
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from pid import simulate, zn_ultimate, tyreus_luyben, simc_pi, pid_tf
P1 = ([0.5], [10, 1])          # thermal plant, K/W
P3 = ([1], [1, 3, 3, 1])       # three equal lags

## Part 1 · Same poles, different zero

*Predict:* which overshoots more, $b$ = 1 or $b$ = 0?

In [2]:
fig, ax = plt.subplots(figsize=(6, 3.2))
for b in (1.0, 0.0):
    t, y, u, _ = simulate(P1, 14, 5, b=b, t_end=40, dt=2e-3)
    ax.plot(t, y, label=f"b = {b:g}")
    print(f"b = {b:g}: overshoot {100 * (y.max() - 1):.2f}%")
ax.set_xlabel("t (s)"); ax.set_ylabel("temperature step response"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

b = 1: overshoot 12.63%


b = 0: overshoot 1.52%


## Part 2 · Tuning rules

In [3]:
Pu = 2 * math.pi / math.sqrt(3)
for name, (Kp, Ti, Td) in (("ZN", zn_ultimate(8, Pu)), ("TL", tyreus_luyben(8, Pu))):
    t, y, _, _ = simulate(P3, Kp, Kp / Ti, Kp * Td, N=10, t_end=60, dt=2e-3)
    t2, y2, _, _ = simulate(P3, Kp, Kp / Ti, Kp * Td, N=10, r=lambda t: 0.0, d=lambda t: 1.0, t_end=60, dt=2e-3)
    print(f"{name}: Kp {Kp:.3f}, Ti {Ti:.3f}, Td {Td:.3f}; overshoot {100 * (y.max() - 1):.1f}%, "
          f"disturbance peak {y2.max():.3f}, IAE {np.trapezoid(np.abs(y2), t2):.3f}")
# SIMC on a first-order-plus-delay fit of 1/(s+1)^3 (half rule: tau = 1.5, theta = 1.5)
Kp, Ti = simc_pi(1, 1.5, 1.5)
t, y, _, _ = simulate(P3, Kp, Kp / Ti, t_end=60, dt=2e-3)
print(f"SIMC PI: Kp {Kp:.3f}, Ti {Ti:.3f}; overshoot {100 * (y.max() - 1):.1f}%")

ZN: Kp 4.800, Ti 1.814, Td 0.453; overshoot 42.7%, disturbance peak 0.189, IAE 0.458


TL: Kp 3.636, Ti 7.981, Td 0.576; overshoot 5.1%, disturbance peak 0.235, IAE 2.191


SIMC PI: Kp 0.500, Ti 1.500; overshoot 5.4%


## Part 3 · Windup

*Predict:* how long does the actuator stay saturated without anti-windup?

In [4]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.2))
for aw in ("none", "back", "clamp"):
    t, y, u, _ = simulate(P1, 14, 5, umin=-10, umax=10, aw=aw, r=lambda t: 4.0, t_end=80, dt=2e-3)
    a1.plot(t, y, label=aw); a2.plot(t, u, label=aw)
    sat = t[u >= 10 - 1e-12]
    print(f"{aw:6s}: peak {y.max():.3f} K, saturated until {sat[-1]:.1f} s")
a1.set_xlabel("t (s)"); a1.set_ylabel("temperature (K)"); a1.legend()
a2.set_xlabel("t (s)"); a2.set_ylabel("actuator power (W)")
plt.show()
if shutil.which("ngspice"):
    out = subprocess.run(["ngspice", "-b", "windup.cir"], capture_output=True, text=True, timeout=120).stdout
    print("ngspice:", [l for l in out.splitlines() if l.startswith(("pk1 =", "pk2 ="))])
else:
    print("ngspice not installed: skip (CI runs windup.cir through tools/run_spice.py)")

none  : peak 4.943 K, saturated until 44.8 s


back  : peak 4.083 K, saturated until 16.1 s


clamp : peak 4.010 K, saturated until 10.7 s
ngspice: ['pk1 = 4.943342e+00', 'pk2 = 4.083444e+00']


## Part 4 · Derivative filter and noise

In [5]:
rng = np.random.default_rng(0)
Kp, Ti, Td = zn_ultimate(8, Pu)
dt = 2e-3
samples = rng.normal(0, 0.01, size=int(30 / dt) + 2)       # sensor noise, 0.01 rms, held for each step
for N in (5, 10, 20):
    t, y, u, _ = simulate(P3, Kp, Kp / Ti, Kp * Td, N=N, c=0, t_end=30, dt=dt,
                          noise=lambda t: samples[int(t / dt)])
    print(f"N = {N:2d}: standard deviation of u after 20 s = {u[t > 20].std():.3f}  (high-frequency gain Kp(1+N) = {Kp * (1 + N):.1f})")

N =  5: standard deviation of u after 20 s = 0.289  (high-frequency gain Kp(1+N) = 28.8)


N = 10: standard deviation of u after 20 s = 0.531  (high-frequency gain Kp(1+N) = 52.8)


N = 20: standard deviation of u after 20 s = 1.019  (high-frequency gain Kp(1+N) = 100.8)


**Discussion.** Answer the four questions in the [lab handout](README.md) before reading its answers.